# 第 3 章：继承与方法解析顺序（MRO）

> 本章目标：掌握**单继承**与方法重写，理解 `super()` 的正确用法，学会分析**多继承**下的 MRO（Method Resolution Order），避开菱形继承的坑。

---

## 3.1 什么是继承？

**继承（Inheritance）**：子类自动获得父类的属性和方法，实现"is-a"（是一个）关系。

- 一只 `Dog` **是一个** `Animal` → 适合继承
- 一辆 `Car` **有一个** `Engine` → 适合组合（见第 7 章）

```mermaid
classDiagram
    class Animal {
        +str name
        +int age
        +eat()
        +sleep()
        +speak()*
    }
    class Dog {
        +speak() "汪汪"
        +fetch()
    }
    class Cat {
        +speak() "喵喵"
        +climb()
    }
    Animal <|-- Dog : 继承
    Animal <|-- Cat : 继承
```

继承的价值：**公共逻辑写一遍，子类各自扩展/覆盖差异部分**。

## 3.2 单继承与方法重写（Override）

- 子类**直接复用**父类的方法（如 `eat`）；
- 子类定义**同名方法**则覆盖父类（如 `speak`）——这叫**重写（Override）**；
- 子类还可以**新增**自己的方法（如 `fetch`）。

In [1]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def eat(self):
        return f"{self.name} 正在吃东西"

    def speak(self):
        return f"{self.name} 发出了声音"


class Dog(Animal):
    def speak(self):                     # 重写父类方法
        return f"{self.name}: 汪汪汪！"

    def fetch(self):                     # 子类新增方法
        return f"{self.name} 把球叼回来了"


dog = Dog("旺财", 3)
print(dog.eat())      # 复用父类的 eat
print(dog.speak())    # 调用子类重写后的 speak
print(dog.fetch())    # 子类独有

旺财 正在吃东西
旺财: 汪汪汪！
旺财 把球叼回来了


## 3.3 `super()`：调用父类的方法

重写时常常需要**先复用父类的逻辑，再补充自己的逻辑**，而不是完全推翻。`super()` 就是干这个的。

```mermaid
sequenceDiagram
    participant D as Dog 实例
    participant S as super
    participant A as Animal
    D->>S: super().__init__(name, age)
    S->>A: 按 MRO 找到父类
    A-->>D: 完成 name/age 初始化
    D->>D: 继续初始化自己的 breed
```

> ⚠️ 常见误解：`super()` 不等于"父类"，它实际是"**按 MRO 顺序找下一个类**"——在单继承里恰好是父类，在多继承里区别很大（见 3.6）。

In [2]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def speak(self):
        return "..."


class Dog(Animal):
    def __init__(self, name, age, breed):
        super().__init__(name, age)   # 父类负责通用部分
        self.breed = breed            # 子类只管自己的特有部分

    def speak(self):
        base = super().speak()        # 复用父类逻辑
        return f"{base} 汪汪汪！（{self.breed}）"


dog = Dog("旺财", 3, "金毛")
print(dog.speak())
print(dog.name, dog.age, dog.breed)

... 汪汪汪！（金毛）
旺财 3 金毛


## 3.4 `isinstance` 与 `issubclass`

| 函数 | 作用 | 注意 |
|------|------|------|
| `isinstance(obj, cls)` | obj 是否是 cls（或其子类）的实例 | ✅ 推荐，考虑继承关系 |
| `type(obj) is cls` | obj 的类型是否**恰好**是 cls | ❌ 不推荐，忽略继承 |
| `issubclass(sub, parent)` | sub 是否是 parent 的子类 | 参数都是类 |

In [3]:
class Animal: pass
class Dog(Animal): pass
class Cat(Animal): pass

dog = Dog()

print(isinstance(dog, Dog))      # True
print(isinstance(dog, Animal))   # True —— 狗"是一个"动物
print(isinstance(dog, Cat))      # False

print(type(dog) is Animal)       # False —— type 是精确匹配

print(issubclass(Dog, Animal))   # True
print(issubclass(Animal, Dog))   # False

# isinstance 第二个参数可以是元组：匹配任意一个
print(isinstance(dog, (Dog, Cat)))  # True

True
True
False
False
True
False
True


## 3.5 多继承与菱形问题

Python 支持**多继承**（一个类继承多个父类），这带来著名的**菱形问题（Diamond Problem）**：

```mermaid
classDiagram
    class A {
        +who()
    }
    class B {
        +who()
    }
    class C {
        +who()
    }
    class D {
    }
    A <|-- B
    A <|-- C
    B <|-- D
    C <|-- D
```

当 `D` 的实例调用 `who()`，而 B、C、A 都有这个方法时——**到底用谁的？**

不同语言的对策：

| 语言 | 多继承支持 | 菱形问题对策 |
|------|-----------|-------------|
| C++ | ✅ | 需要 `virtual` 虚继承，手工解决 |
| Java | ❌（类只能单继承，接口可多实现） | 从语法上回避 |
| Python | ✅ | **MRO（方法解析顺序）** 自动决定 |

## 3.6 MRO：C3 线性化算法

Python 用 **C3 线性化**算法把继承关系"拉直"成一条**有序的查找链**，称为 MRO。规则保证：

1. **子类永远排在父类前面**；
2. **继承声明的顺序保持**（`class D(B, C)` 中 B 在 C 前）；
3. **单调性**：若 X 在 Y 前出现在某类的 MRO 中，其子类的 MRO 也保持这个顺序。

查看方法：
- `类名.__mro__` → 元组
- `类名.mro()` → 列表

```mermaid
flowchart TD
    D[D 的实例调用 who] --> E[按 MRO 顺序查找]
    E --> F1["1. D 自己有吗?"]
    F1 -- 否 --> F2["2. B 有吗?"]
    F2 -- 有 --> G[✅ 使用 B.who]
    F2 -- 否 --> F3["3. C 有吗?"]
    F3 -- 否 --> F4["4. A 有吗?"]
    F4 -- 否 --> F5[5. object → AttributeError]
```

In [4]:
class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B"

class C(A):
    def who(self):
        return "C"

class D(B, C):
    pass


d = D()
print(d.who())          # B —— MRO 中 B 排在最前

# 查看完整的查找链
for cls in D.__mro__:
    print(cls.__name__, end=" → ")
print("结束")
# D → B → C → A → object
# 注意：C 在 A 前面！这不是简单的"深度优先"

B
D → B → C → A → object → 结束


## 3.7 协作式多继承：`super()` 的进阶用法

多继承中，如果每个类的 `__init__` 都调用 `super().__init__()`，MRO 链条上的**每个类都会被恰好初始化一次**：

```mermaid
flowchart LR
    D -->|super| B -->|super| C -->|super| A -->|super| object
    style D fill:#e1f5ff
    style object fill:#ffe1e1
```

这就是为什么 `super()` 不叫 "parent"——在 D 的方法里，`super()` 指向的是 **B**（MRO 的下一个），而在 B 的方法里指向的是 **C**，不是 A！

**协作式多继承的规矩**：
1. 每个类都用 `super().__init__(**kwargs)` 把参数往后传；
2. 每个类只取走自己认识的参数；
3. 顶层基类（A）做好收尾。

In [5]:
class A:
    def __init__(self, a_param, **kwargs):
        super().__init__(**kwargs)   # 链条末端，kwargs 应为空
        self.a_param = a_param
        print(f"A 初始化: {a_param}")

class B(A):
    def __init__(self, b_param, **kwargs):
        super().__init__(**kwargs)   # 把剩余参数传给 MRO 下一个（C！）
        self.b_param = b_param
        print(f"B 初始化: {b_param}")

class C(A):
    def __init__(self, c_param, **kwargs):
        super().__init__(**kwargs)
        self.c_param = c_param
        print(f"C 初始化: {c_param}")

class D(B, C):
    def __init__(self, d_param, **kwargs):
        super().__init__(**kwargs)
        self.d_param = d_param
        print(f"D 初始化: {d_param}")


print("MRO:", " → ".join(c.__name__ for c in D.__mro__))
print("---")
d = D(d_param="d", b_param="b", c_param="c", a_param="a")
# 输出顺序正好是 MRO 顺序的反向：A → C → B → D（先 super 后自己）

MRO: D → B → C → A → object
---
A 初始化: a
C 初始化: c
B 初始化: b
D 初始化: d


## 3.8 ⚠️ 多继承实践建议

1. **优先单继承**：层次清晰，问题少；
2. 多继承只用于 **Mixin**（只提供方法、不保存状态的小类，见第 7 章）；
3. 如果 MRO 无法满足线性化，Python 会直接**报错**拒绝创建类：

In [6]:
# 一个无法线性化的例子（继承顺序自相矛盾）
class X: pass
class Y(X): pass

try:
    class Z(X, Y):   # X 在 Y 前，但 Y 又是 X 的子类 → 矛盾
        pass
except TypeError as e:
    print(f"MRO 冲突: {e}")

MRO 冲突: Cannot create a consistent method resolution
order (MRO) for bases X, Y


## 3.9 本章小结

| 概念 | 一句话记忆 |
|------|-----------|
| 继承 | "is-a" 关系，复用父类代码 |
| 重写 Override | 子类同名方法覆盖父类 |
| `super()` | 找 MRO 的**下一个**类，不一定是父类 |
| `isinstance` | 判断类型要考虑继承关系 |
| MRO | C3 线性化，子类在前、声明顺序保持 |
| `__mro__` | 查看完整查找链的调试利器 |

### 📝 动手练习

1. 设计 `Vehicle` → `Car` → `ElectricCar` 三层继承，`ElectricCar` 重写 `refuel()` 为 `charge()`，并用 `super().__init__()` 复用上层初始化。
2. 构造一个菱形继承结构，打印 `__mro__`，预测每个方法的调用结果后再运行验证。
3. 思考题：为什么 Python 3 中所有类的 MRO 都以 `object` 结尾？（提示：新式类统一继承 object）

---
**下一章** 👉 `04_多态与鸭子类型.ipynb`：学习"同一接口、不同实现"的多态思想，以及 Python 特有的鸭子类型哲学。